# UCS547 – Accelerated Data Science
## Assignment 2: CUDA C++ Programming  

**Name:** Ajay Goyal  
**Roll Number:** 102303197


## **Q1. Identify !, %, and %% in Google Colab**

### Bang Operator (!)

The bang operator (!) is used to run Linux shell commands directly from a Colab code cell.

It allows us to execute terminal commands such as checking CUDA version, listing files, etc.

Example: Check CUDA compiler version using nvcc.


In [2]:
!nvidia-smi
!nvcc --version



Fri Feb 13 07:48:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Line Magic Command (%)

The percent symbol (%) is used for line magic commands. These commands apply to only one line.

They are used for timing, environment management, and debugging.

Example: Measure execution time of a statement using %time.


In [3]:
%time sum(range(1000000))


CPU times: user 18.1 ms, sys: 15 µs, total: 18.1 ms
Wall time: 18.2 ms


499999500000

### Cell Magic Command (%%)

The double percent symbol (%%) is used for cell magic commands. These commands apply to the entire cell.

They are useful for timing, writing CUDA code, or running bash scripts.

Example: Measure execution time of entire cell using %%time.


In [4]:
%%time

total = 0
for i in range(1000000):
    total += i

print("Sum:", total)


Sum: 499999500000
CPU times: user 110 ms, sys: 0 ns, total: 110 ms
Wall time: 110 ms


## **Q2. Identify key `nvidia-smi` commands with multiple options**

1. Check GPU status:

In [5]:
!nvidia-smi

Fri Feb 13 07:53:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

2. Show detailed GPU information:

In [7]:
!nvidia-smi -q



==============NVSMI LOG==============

Timestamp                                 : Fri Feb 13 07:54:28 2026
Driver Version                            : 580.82.07
CUDA Version                              : 13.0

Attached GPUs                             : 1
GPU 00000000:00:04.0
    Product Name                          : Tesla T4
    Product Brand                         : NVIDIA
    Product Architecture                  : Turing
    Display Mode                          : Requested functionality has been deprecated
    Display Attached                      : Yes
    Display Active                        : Disabled
    Persistence Mode                      : Disabled
    Addressing Mode                       : None
    MIG Mode
        Current                           : N/A
        Pending                           : N/A
    Accounting Mode                       : Disabled
    Accounting Mode Buffer Size           : 4000
    Driver Model
        Current                           : N/

3. Show only utilization:

In [8]:
!nvidia-smi --query-gpu=utilization.gpu --format=csv


utilization.gpu [%]
0 %


4. Show memory usage:

In [9]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


memory.used [MiB], memory.total [MiB]
0 MiB, 15360 MiB


# **Q.3 Debug common CUDA errors (zero output, incorrect indexing, PTX errors)**



1. **Zero Output**
   - Cause: The GPU kernel may not finish execution before the CPU continues, because CUDA execution is asynchronous.
   - Fix: Use cudaDeviceSynchronize() to ensure GPU finishes execution before CPU proceeds.
   ```cpp
   cudaDeviceSynchronize();


2. **Incorrect Indexing**
   - Cause: Using only threadIdx.x accesses the thread index within a single block and does not account for multiple blocks, which can lead to incorrect data access or repeated processing of the same elements.
   - Fix: Compute the global thread ID using blockIdx.x, blockDim.x, and threadIdx.x to uniquely identify each thread across all blocks.
   ```cpp
   int global_thread_id = blockIdx.x * blockDim.x + threadIdx.x;




3. **PTX Error**
   - Cause: PTX errors occur when the CUDA program is compiled for an incompatible or unsupported GPU architecture.
   - Fix: Compile the CUDA program using the correct architecture flag that matches the GPU.
   ```bash
   nvcc -arch=sm_75 program.cu

# Q.4 Write a CUDA C/C++ program to demonstrate GPU kernel execu'on and thread indexing.
**a. Launch a CUDA kernel using: 1 block and 8 threads**

**b. Each thread must print: Hello from GPU thread <global_thread_id>**

**c. Compute the global thread ID using: global_thread_id = blockIdx.x * blockDim.x + threadIdx.x**

**d. Clearly separate: Host code (CPU) & Device code (GPU kernel)**

In [10]:
%%writefile thread_indexing.cu
#include <stdio.h>

// Device Code (GPU Kernel)
// This runs on GPU


__global__ void helloKernel()
{
    // Compute global thread ID
    int global_thread_id = blockIdx.x * blockDim.x + threadIdx.x;

    // Each thread prints its ID
    printf("Hello from GPU thread %d\n", global_thread_id);
}


// Host Code (CPU Main Function)
// This runs on CPU


int main()
{
    printf("Host: Launching GPU kernel with 1 block and 8 threads...\n");

    // Launch kernel
    helloKernel<<<1, 8>>>();

    // Wait for GPU to finish
    cudaDeviceSynchronize();

    printf("Host: Kernel execution completed.\n");

    return 0;
}


Writing thread_indexing.cu


In [11]:
!nvcc -arch=sm_75 thread_indexing.cu -o thread_indexing
!./thread_indexing


Host: Launching GPU kernel with 1 block and 8 threads...
Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3
Hello from GPU thread 4
Hello from GPU thread 5
Hello from GPU thread 6
Hello from GPU thread 7
Host: Kernel execution completed.


#**Q5. CUDA Program to demonstrate Host and Device Memory Separation**

This program demonstrates the separation between Host (CPU) memory and Device (GPU) memory.

Steps performed:

1. Create an integer array of size 5 on the Host (CPU)
2. Allocate memory on the Device (GPU) using cudaMalloc()
3. Copy data from Host to Device using cudaMemcpy()
4. Launch a GPU kernel where each thread prints values from Device memory
5. Copy data back from Device to Host and print on CPU

This shows that CPU and GPU have separate memory spaces and data must be explicitly copied.


In [12]:
%%writefile memory_separation.cu
#include <stdio.h>

// Device Code (GPU Kernel)
// Runs on GPU and accesses device memory

__global__ void printFromDevice(int *device_array)
{
    int idx = threadIdx.x;

    printf("GPU Thread %d reads value = %d\n", idx, device_array[idx]);
}

// Host Code (CPU Main Function)
// Runs on CPU and manages memory transfer

int main()
{
    // Step a: Create array on Host (CPU)
    int host_array[5] = {10, 20, 30, 40, 50};

    // Pointer for Device memory
    int *device_array;

    printf("Host: Original data in CPU memory:\n");
    for(int i = 0; i < 5; i++)
    {
        printf("%d ", host_array[i]);
    }
    printf("\n\n");

    // Step b: Allocate memory on Device (GPU)
    cudaMalloc((void**)&device_array, 5 * sizeof(int));

    // Step c: Copy data from Host to Device
    cudaMemcpy(device_array, host_array, 5 * sizeof(int), cudaMemcpyHostToDevice);

    // Step d: Launch kernel (1 block, 5 threads)
    printFromDevice<<<1, 5>>>(device_array);

    // Wait for GPU to finish
    cudaDeviceSynchronize();

    // Step e: Copy data back from Device to Host
    cudaMemcpy(host_array, device_array, 5 * sizeof(int), cudaMemcpyDeviceToHost);

    printf("\nHost: Data copied back to CPU memory:\n");
    for(int i = 0; i < 5; i++)
    {
        printf("%d ", host_array[i]);
    }
    printf("\n");

    // Free Device memory
    cudaFree(device_array);

    return 0;
}


Writing memory_separation.cu


In [13]:
!nvcc -arch=sm_75 memory_separation.cu -o memory_separation
!./memory_separation


Host: Original data in CPU memory:
10 20 30 40 50 

GPU Thread 0 reads value = 10
GPU Thread 1 reads value = 20
GPU Thread 2 reads value = 30
GPU Thread 3 reads value = 40
GPU Thread 4 reads value = 50

Host: Data copied back to CPU memory:
10 20 30 40 50 


### **Q6. Compare CPU times of List, Tuple, and NumPy Array**

This experiment compares the execution time required to compute the sum of elements using:

- Python List
- Python Tuple
- NumPy Array

The time module is used to measure CPU execution time.

NumPy arrays are faster because they use optimized C-based implementation and vectorized operations, while Lists and Tuples use slower Python loops.


In [16]:
import time
import numpy as np

# Number of elements
N = 10000000

# Create List
python_list = list(range(N))

# Create Tuple
python_tuple = tuple(range(N))

# Create NumPy Array
numpy_array = np.arange(N)

# Measure List time
start = time.time()
sum_list = sum(python_list)
end = time.time()
print("List sum:", sum_list)
print("List CPU time:", end - start, "seconds\n")

# Measure Tuple time
start = time.time()
sum_tuple = sum(python_tuple)
end = time.time()
print("Tuple sum:", sum_tuple)
print("Tuple CPU time:", end - start, "seconds\n")

# Measure NumPy time
start = time.time()
sum_numpy = np.sum(numpy_array)
end = time.time()
print("NumPy sum:", sum_numpy)
print("NumPy CPU time:", end - start, "seconds\n")


List sum: 49999995000000
List CPU time: 0.07961797714233398 seconds

Tuple sum: 49999995000000
Tuple CPU time: 0.07716584205627441 seconds

NumPy sum: 49999995000000
NumPy CPU time: 0.006717681884765625 seconds



### **Conclusion**

From the above experiment, NumPy array takes less CPU time compared to List and Tuple.

This is because NumPy uses optimized C-level implementation and vectorized operations, which makes it much faster for numerical computations.

Therefore, NumPy is preferred for high-performance scientific computing.
